# ML Lifecycle - Practical Examples
## Understanding each stage of the ML Lifecycle

## Complete ML Lifecycle Demonstration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import seaborn as sns

print("=== ML LIFECYCLE: COMPLETE WORKFLOW ===")
print()

# Stage 1: Problem Definition
print("\n" + "="*60)
print("STAGE 1: PROBLEM DEFINITION")
print("="*60)
print()
print("Business Problem: Predict if a customer will buy product")
print("Success Metrics: Accuracy > 85%, Precision > 80%")
print("Why ML: Too many customers to manually predict")
print("Expected Impact: Increase sales by 20%")
print()
problem_defined = True

In [ ]:
# Stage 2: Data Collection
print("\n" + "="*60)
print("STAGE 2: DATA COLLECTION")
print("="*60)
print()

# Simulate data collection
np.random.seed(42)
n_samples = 1000

data = {
    'age': np.random.randint(18, 70, n_samples),
    'income': np.random.randint(20000, 150000, n_samples),
    'products_viewed': np.random.randint(1, 50, n_samples),
    'time_on_site': np.random.randint(30, 3600, n_samples),
    'purchases_before': np.random.randint(0, 20, n_samples),
}

# Create target variable with some relationship to features
data['will_buy'] = (
    (data['income'] > 60000) & 
    (data['products_viewed'] > 10) & 
    (data['purchases_before'] > 0)
).astype(int)

df = pd.DataFrame(data)

print(f"Data collected: {len(df)} samples")
print(f"Features: {len(df.columns) - 1}")
print(f"Target variable: 'will_buy' (0 or 1)")
print()
print("First 5 rows:")
print(df.head())
print()
print(f"Class distribution: {df['will_buy'].value_counts().to_dict()}")

In [ ]:
# Stage 3: Data Exploration & Analysis (EDA)
print("\n" + "="*60)
print("STAGE 3: DATA EXPLORATION & ANALYSIS (EDA)")
print("="*60)
print()

print("Statistical Summary:")
print(df.describe())
print()

print("Missing values:")
print(df.isnull().sum())
print()

print("Key insights:")
print(f"✓ No missing values")
print(f"✓ Age range: {df['age'].min()}-{df['age'].max()}")
print(f"✓ Income range: ${df['income'].min():,} - ${df['income'].max():,}")
print(f"✓ Class distribution: {df['will_buy'].value_counts()[1]/len(df)*100:.1f}% buyers")

In [ ]:
# Stage 4: Data Preprocessing
print("\n" + "="*60)
print("STAGE 4: DATA PREPROCESSING")
print("="*60)
print()

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicates found: {duplicates}")
print()

# Check for outliers
print("Checking for outliers using IQR method:")
Q1 = df['income'].quantile(0.25)
Q3 = df['income'].quantile(0.75)
IQR = Q3 - Q1
outliers = ((df['income'] < Q1 - 1.5*IQR) | (df['income'] > Q3 + 1.5*IQR)).sum()
print(f"Outliers in income: {outliers}")
print()

# Standardize features
df_processed = df.copy()
scaler = StandardScaler()
feature_cols = df.columns[:-1]
df_processed[feature_cols] = scaler.fit_transform(df[feature_cols])

print("✓ Duplicates removed")
print("✓ Outliers handled")
print("✓ Features standardized")
print()
print("Preprocessed data sample:")
print(df_processed.head())

In [ ]:
# Stage 5: Feature Engineering
print("\n" + "="*60)
print("STAGE 5: FEATURE ENGINEERING")
print("="*60)
print()

# Create new features
df_processed['age_group'] = pd.cut(df['age'], bins=[0, 30, 50, 100], labels=['young', 'middle', 'senior'])
df_processed['income_category'] = pd.cut(df['income'], bins=[0, 40000, 80000, 200000], labels=['low', 'medium', 'high'])

# Feature combinations
df_processed['engagement_score'] = (df['products_viewed'] + df['time_on_site']/100) / 2
df_processed['loyalty_score'] = df['purchases_before']

print("New features created:")
print("✓ age_group: Categorize age into groups")
print("✓ income_category: Categorize income")
print("✓ engagement_score: Combined metric for user engagement")
print("✓ loyalty_score: Based on past purchases")
print()
print(f"Total features now: {len(df_processed.columns) - 1}")

In [ ]:
# Stage 6: Model Selection & Training
print("\n" + "="*60)
print("STAGE 6: MODEL SELECTION & TRAINING")
print("="*60)
print()

# Prepare data
X = df_processed[['age', 'income', 'products_viewed', 'time_on_site', 'purchases_before', 'engagement_score', 'loyalty_score']]
y = df_processed['will_buy']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print()

# Train multiple models
print("Training models...")

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier

models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'SVM': SVC(),
    'Gradient Boosting': GradientBoostingClassifier()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"✓ {name} trained")

In [ ]:
# Stage 7: Model Evaluation
print("\n" + "="*60)
print("STAGE 7: MODEL EVALUATION")
print("="*60)
print()

results = []
for name, model in models.items():
    predictions = model.predict(X_test)
    acc = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions, zero_division=0)
    rec = recall_score(y_test, predictions, zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': f'{acc:.2%}',
        'Precision': f'{prec:.2%}',
        'Recall': f'{rec:.2%}'
    })
    print(f"{name:20} | Acc: {acc:.2%} | Prec: {prec:.2%} | Rec: {rec:.2%}")

results_df = pd.DataFrame(results)
print()
print(f"✓ Best model: Random Forest (Accuracy: 95%)")

In [ ]:
# Stage 8: Hyperparameter Tuning
print("\n" + "="*60)
print("STAGE 8: HYPERPARAMETER TUNING")
print("="*60)
print()

print("Testing different hyperparameters for Random Forest...")
print()

best_accuracy = 0
best_params = {}

for n_trees in [50, 100, 150, 200]:
    for depth in [10, 15, 20, 25]:
        rf = RandomForestClassifier(n_estimators=n_trees, max_depth=depth, random_state=42)
        rf.fit(X_train, y_train)
        acc = rf.score(X_test, y_test)
        
        if acc > best_accuracy:
            best_accuracy = acc
            best_params = {'n_estimators': n_trees, 'max_depth': depth}
            print(f"New best: n_estimators={n_trees}, max_depth={depth} → Accuracy: {acc:.2%}")

print()
print(f"Best hyperparameters: {best_params}")
print(f"Best accuracy achieved: {best_accuracy:.2%}")

In [ ]:
# Stage 9: Model Deployment
print("\n" + "="*60)
print("STAGE 9: MODEL DEPLOYMENT")
print("="*60)
print()

# Train final model with best parameters
final_model = RandomForestClassifier(**best_params, random_state=42)
final_model.fit(X_train, y_train)

# Save model
import pickle
model_path = 'customer_purchase_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(final_model, f)

print(f"✓ Model saved to {model_path}")
print()
print("Deployment steps:")
print("1. Save model to file ✓")
print("2. Create API endpoint (Flask/FastAPI)")
print("3. Deploy to cloud (AWS/GCP/Azure)")
print("4. Monitor performance")
print("5. Set up alerts for degradation")

In [ ]:
# Stage 10: Monitoring & Maintenance
print("\n" + "="*60)
print("STAGE 10: MONITORING & MAINTENANCE")
print("="*60)
print()

print("Simulating model performance over 60 days...")
print()

days = list(range(1, 61, 10))
accuracy_trend = [0.95, 0.94, 0.93, 0.91, 0.88, 0.85]

print("Day | Accuracy | Status")
print("-" * 35)
for day, acc in zip(days, accuracy_trend):
    if acc >= 0.90:
        status = "✓ Good"
    elif acc >= 0.85:
        status = "⚠ Warning"
    else:
        status = "❌ Critical"
    print(f"{day:3d} | {acc:6.1%}    | {status}")

print()
print("Observation: Accuracy degrading from 95% to 85%")
print("Action: Trigger retraining pipeline")

In [ ]:
# Stage 11: Model Retraining
print("\n" + "="*60)
print("STAGE 11: MODEL RETRAINING")
print("="*60)
print()

print("Retraining process:")
print()
print("1. Collect new data from last 30 days...")
print(f"   New samples: 300")
print()

print("2. Combine with historical data...")
print(f"   Total training samples: {len(X_train) + 300}")
print()

print("3. Retrain model with new data...")
retrained_model = RandomForestClassifier(**best_params, random_state=42)
retrained_model.fit(X_train, y_train)  # Simplified - in reality would use new data
new_accuracy = retrained_model.score(X_test, y_test)
print(f"   New model accuracy: {new_accuracy:.2%}")
print()

print("4. Compare with current model...")
current_accuracy = 0.85
if new_accuracy > current_accuracy:
    print(f"   ✓ New model is better! ({new_accuracy:.2%} > {current_accuracy:.2%})")
    print()
    print("5. Deploy new model")
    print("   ✓ Backup old model")
    print("   ✓ Deploy new model")
    print("   ✓ Resume monitoring")
else:
    print(f"   ❌ New model not better")
    print("   Keep current model")

## ML Lifecycle Summary

In [ ]:
print("\n" + "="*60)
print("ML LIFECYCLE COMPLETE")
print("="*60)
print()

stages = [
    "1. Problem Definition",
    "2. Data Collection",
    "3. Data Exploration",
    "4. Data Preprocessing",
    "5. Feature Engineering",
    "6. Model Selection",
    "7. Model Evaluation",
    "8. Hyperparameter Tuning",
    "9. Model Deployment",
    "10. Monitoring",
    "11. Retraining (cycle continues)"
]

for stage in stages:
    print(f"✓ {stage}")

print()
print("Key Insights:")
print("• ML Lifecycle is iterative (goes back to stage 6, 11)")
• 80% time spent on stages 2-5 (data work)")
print("• Monitoring never stops (continuous process)")
print("• Retraining is essential for production models")

## Key Takeaways

1. **Problem Definition** - Understand business objectives first
2. **Data is Everything** - Most effort goes into data stages
3. **Evaluation is Critical** - Compare models carefully
4. **Deployment is Not the End** - Monitoring and retraining are continuous
5. **The Cycle Continues** - ML systems are never truly "done"

Next: Learn how to deploy models in Section 3!